# Asignación Automatizada de Turnos Médicos con LLMs Open-Weight (<8B)
## Entrega 1 - Cuaderno de Factibilidad y Demostración de Fallo de Línea Base

Este cuaderno demuestra:
1. **Factibilidad de Ejecución** en GPU gratuita de Google Colab (Nvidia T4 de 15 GB VRAM) o hardware local.
2. Carga del Modelo Candidato (`Qwen/Qwen2.5-3B-Instruct` o `meta-llama/Llama-3.2-3B-Instruct`) en precisión cuantizada de 4-bit (NF4) / 16-bit FP16.
3. Verificación empírica del **Fallo en Prompting Directo (Zero-Shot)** frente al `ScheduleVerifier`.

In [ ]:
# Paso 1: Instalación de dependencias
!pip install -q transformers accelerate bitsandbytes torch

In [ ]:
# Paso 2: Configuración de caché y verificación de dispositivo GPU
import os
import torch

# Garantizar directorio de caché con permisos de escritura
hf_cache = os.path.expanduser("~/.cache/huggingface")
try:
    os.makedirs(hf_cache, exist_ok=True)
    with open(os.path.join(hf_cache, ".test"), "w") as f:
        f.write("ok")
    os.remove(os.path.join(hf_cache, ".test"))
except (PermissionError, OSError):
    hf_cache = os.path.expanduser("~/.hf_cache")
    os.makedirs(hf_cache, exist_ok=True)
    os.environ["HF_HOME"] = hf_cache
    print(f"Redirigiendo caché de HuggingFace a: {hf_cache}")

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import json

print(f"CUDA Disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Dispositivo GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM Total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
elif torch.backends.mps.is_available():
    print("Dispositivo: Apple Silicon GPU (MPS disponible)")
else:
    print("Dispositivo: CPU (Para inferencia acelerada, selecciona GPU en Entorno de Ejecución en Colab o CUDA local)")

In [ ]:
# Paso 3: Carga del Modelo (Detección automática: 4-bit NF4 en GPU CUDA / nativo en Apple Silicon o CPU)
model_id = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

if torch.cuda.is_available():
    # En Google Colab / GPU Nvidia: Cuantización 4-bit NF4 con bitsandbytes (VRAM ~2.2 - 2.4 GB)
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto"
    )
    allocated = torch.cuda.memory_allocated() / 1e9
    print(f"¡Modelo cargado exitosamente en GPU (4-bit NF4)! VRAM asignada: {allocated:.2f} GB / 15.0 GB (Colab T4)")
elif torch.backends.mps.is_available():
    # En Mac con Apple Silicon (GPU MPS)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        device_map="auto"
    )
    print("¡Modelo cargado exitosamente en Apple Silicon GPU (MPS) en FP16!")
else:
    # En CPU local (o entorno sin GPU dedicada)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float32,
        low_cpu_mem_usage=True
    )
    print("¡Modelo cargado exitosamente en CPU local!")

In [ ]:
# Paso 4: Inferencia de Línea Base mediante Prompting Directo (Zero-Shot)
prompt = """Eres un asistente automatizado para la asignación de turnos médicos hospitalarios.
Asigna médicos a los turnos de toda la semana en formato JSON válido sin violar:
1. Mínimo 16h de descanso continuo tras turnos de Noche.
2. Máximo 40h semanales por médico.
3. Al menos 1 Anestesiólogo en turnos de Noche y 1 Urgenciólogo en turnos de Mañana/Tarde.
Retorna ÚNICAMENTE JSON válido."""

device = model.device if hasattr(model, "device") else ("cuda" if torch.cuda.is_available() else "cpu")
messages = [{"role": "user", "content": prompt}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True
).to(device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.2,
        pad_token_id=tokenizer.eos_token_id
    )

prompt_len = inputs["input_ids"].shape[1]
response = tokenizer.decode(outputs[0][prompt_len:], skip_special_tokens=True)
print("=== SALIDA CRUDA DEL PROMPTING DIRECTO ===\n", response)
